In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, explode_outer, md5, concat_ws, lit, to_json
from pyspark.sql.types import StructType, ArrayType

def appiattisci_json_relazionale(df: DataFrame, chiavi_padre: list = [], nome_livello_precedente: str = "root") -> dict:
    
    tabelle_generate = {}
    chiavi_locali = list(chiavi_padre)
    
    struttura_modificata = True
    while struttura_modificata:
        struttura_modificata = False
        nuove_colonne = []
        for campo in df.schema.fields:
            if campo.name in chiavi_locali:
                nuove_colonne.append(col(campo.name))
            elif isinstance(campo.dataType, StructType):
                struttura_modificata = True
                for sotto_campo in campo.dataType.fields:
                    nuovo_nome = f"{campo.name}_{sotto_campo.name}"
                    nuove_colonne.append(col(f"{campo.name}.{sotto_campo.name}").alias(nuovo_nome))
                    
                    if ("id" in sotto_campo.name.lower()) and nuovo_nome not in chiavi_locali:
                        chiavi_locali.append(nuovo_nome)
            else:
                nuove_colonne.append(col(campo.name))
        if struttura_modificata:
            df = df.select(nuove_colonne)

    schema = df.schema
    campi_da_mantenere = []
    array_da_separare = {} 
    
    for campo in schema.fields:
        nome_campo = campo.name
        if nome_campo in chiavi_locali:
            continue
            
        if isinstance(campo.dataType, ArrayType):
            array_da_separare[nome_campo] = campo.dataType
        else:
            campi_da_mantenere.append(col(nome_campo))
            
    colonne_finali_correnti = [col(k) for k in chiavi_locali] + campi_da_mantenere
    tabelle_generate[nome_livello_precedente] = df.select(colonne_finali_correnti).dropDuplicates()

    for nome_campo, tipo_dato in array_da_separare.items():
        
        chiavi_per_figlio = list(chiavi_locali)
        ha_id_nativo = any('id' in k.lower() for k in chiavi_per_figlio)
        for campo_corrente in df.schema.fields:
            if ("id" in campo_corrente.name.lower()):
                if campo_corrente.name not in chiavi_per_figlio:
                    chiavi_per_figlio.append(campo_corrente.name)

        tipo_elemento = tipo_dato.elementType
        
        df_esploso = df.select(
            *[col(k) for k in chiavi_per_figlio], 
            explode_outer(col(nome_campo)).alias("elemento")
        )
        
        if not ha_id_nativo:
            nome_id_figlio = f"{nome_campo}_id"
            if isinstance(tipo_elemento, StructType):
                df_esploso = df_esploso.withColumn(nome_id_figlio, md5(concat_ws("_", lit(nome_campo), to_json(col("elemento")))))
            else:
                df_esploso = df_esploso.withColumn(nome_id_figlio, md5(concat_ws("_", lit(nome_campo), col("elemento").astype("string"))))
            chiavi_per_figlio.append(nome_id_figlio)

        if isinstance(tipo_elemento, StructType):
            df_figlio = df_esploso.select(*[col(k) for k in chiavi_per_figlio], "elemento.*")
            
            for f in tipo_elemento.fields:
                if ("id" in f.name.lower()) and f.name not in chiavi_per_figlio:
                    chiavi_per_figlio.append(f.name)
        else:
            df_figlio = df_esploso.select(*[col(k) for k in chiavi_per_figlio], col("elemento").alias("elemento"))
        
        df_figlio = df_figlio.dropDuplicates()
                
        sotto_tabelle = appiattisci_json_relazionale(df_figlio, chiavi_per_figlio, nome_campo)
        for sotto_nome, sotto_df in sotto_tabelle.items():
            if sotto_nome == "root":
                tabelle_generate[nome_campo] = sotto_df
            else:
                tabelle_generate[sotto_nome] = sotto_df
            
    return tabelle_generate

In [0]:
#this code is used to generate a unique normalized table from a table with two columns: col id and json field
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, explode_outer, from_json, schema_of_json, lit
from pyspark.sql.types import StructType, ArrayType

def appiattisci_a_tabella_unica_easy(df: DataFrame, colonna_id: str, colonna_json: str) -> DataFrame:

    campionario_json = df.select(colonna_json).first()[0]
    schema_ddl = df.select(schema_of_json(lit(campionario_json))).first()[0]
    df = df.withColumn("json_struct", from_json(col(colonna_json), schema_ddl))
    df = df.select(col(colonna_id), "json_struct.*")

    ancora_da_appiattire = True
    while ancora_da_appiattire:
        ancora_da_appiattire = False
        nuove_colonne = []
        
        for campo in df.schema.fields:
            if campo.name == colonna_id:
                nuove_colonne.append(col(campo.name))
                continue
                
            if isinstance(campo.dataType, StructType):
                ancora_da_appiattire = True
                for sotto_campo in campo.dataType.fields:
                    nuove_colonne.append(col(f"{campo.name}.{sotto_campo.name}").alias(f"{campo.name}_{sotto_campo.name}"))
            
            elif isinstance(campo.dataType, ArrayType):
                ancora_da_appiattire = True
                df = df.withColumn(campo.name, explode_outer(col(campo.name)))
                nuove_colonne = [col(c.name) for c in df.schema.fields]
                break
            else:
                nuove_colonne.append(col(campo.name))
                
        df = df.select(nuove_colonne)
        
    return df

In [0]:
dati_test = [
    (
        "REC_001", 
        '{"data_ordine": "2026-07-08", "info_negozio": {"citta": "Milano", "codice_store": "MI01"}, "prodotti": [{"prod_id": "P10", "prezzo": 150.0, "tag": ["tech", "smart"]}, {"prod_id": "P20", "prezzo": 25.0, "tag": ["home"]}]}'
    ),
    (
        "REC_002", 
        '{"data_ordine": "2026-07-09", "info_negozio": {"citta": "Roma", "codice_store": "RM02"}, "prodotti": [{"prod_id": "P10", "prezzo": 140.0, "tag": ["tech"]}]}'
    )
]

df_input = spark.createDataFrame(dati_test, ["record_id", "payload_json"])

In [0]:
#to be used if the id is in the file name
import os
from pyspark.sql.functions import col, split, element_at, current_timestamp, input_file_name, from_json, schema_of_json, lit

CATALOGO_TARGET = "workspace"
SCHEMA_TARGET = "just_trial"
PREFISSO_TABELLE = "fact_stats"

PATH_VOLUME_ROOT_RECURSIVE = "/Volumes/workspace/just_trial/factory_raw_data"

print("1. Scansione globale del Volume alla ricerca di file JSON con Unity Catalog...")
tabella_bronze = f"{CATALOGO_TARGET}.{SCHEMA_TARGET}.{PREFISSO_TABELLE}_bronze_staging"

try:
    file_gia_caricati = set()
    if spark.catalog.tableExists(tabella_bronze):
        print(f"Estrazione file già elaborati dalla tabella {tabella_bronze}...")
        df_log = spark.table(tabella_bronze).select("nome_file_origine").distinct()
        file_gia_caricati = set([row.nome_file_origine for row in df_log.collect()])
    
    all_files_df = spark.read.format("text").load(PATH_VOLUME_ROOT_RECURSIVE) \
        .select(element_at(split(col("_metadata.file_path"), "/"), -1).alias("nome_file_completo"), col("_metadata.file_path").alias("percorso_completo")) \
        .distinct()
    
    lista_totale_files = all_files_df.collect()
    percorsi_file_nuovi = [row.percorso_completo for row in lista_totale_files if row.nome_file_completo not in file_gia_caricati]
    
    if not percorsi_file_nuovi:
        print("Nessun file nuovo trovato nell'intero storico del Volume. Il database è già aggiornato.")
    else:
        print(f"Trovati {len(percorsi_file_nuovi)} file nuovi/arretrati da elaborare!")
        
        df_nuovi_raw = spark.read.format("text").option("wholetext", "true").load(percorsi_file_nuovi)
        
        df_staging = df_nuovi_raw \
            .withColumn("nome_file_origine", element_at(split(input_file_name(), "/"), -1)) \
            .withColumn("record_id", col("nome_file_origine")) \
            .withColumnRenamed("value", "payload_json") \
            .withColumn("file_loading_dtm", current_timestamp())
            
        
        campionario_json = df_staging.select("payload_json").first()[0]
        schema_ddl = df_staging.select(schema_of_json(lit(campionario_json))).first()[0]

        df_strutturato = df_staging.withColumn("json_struct", from_json(col("payload_json"), schema_ddl))
        df_pronto_per_ricorsione = df_strutturato.select(col("record_id"), "json_struct.*")

        dizionario_tabelle = appiattisci_json_relazionale(
            df_pronto_per_ricorsione, 
            chiavi_padre=["record_id"], 
            nome_livello_precedente="root"
        )

        print("4. Scrittura delle tabelle relazionali in Unity Catalog...")
        for nome_livello, df_risultato in dizionario_tabelle.items():
            nome_tabella_target = f"{CATALOGO_TARGET}.{SCHEMA_TARGET}.{PREFISSO_TABELLE}_{nome_livello}"
            
            print(f"Aggiornamento tabella relazionale: {nome_tabella_target}")
            
            df_risultato.write \
                .format("delta") \
                .mode("append") \
                .option("mergeSchema", "true") \
                .saveAsTable(nome_tabella_target)
                
        print("Tutte le tabelle padre e figlie sono state allineate con successo!")

except Exception as e:
    print(f"Errore critico durante l'elaborazione globale: {str(e)}")

print("-" * 60)
print("Processo di allineamento storico completato!")

In [0]:
from pyspark.sql.functions import col, element_at, split, current_timestamp
import os

CATALOGO_TARGET = "workspace"
SCHEMA_TARGET = "just_trial"
PREFISSO_TABELLE = "fact_stats"

PATH_VOLUME_ROOT_RECURSIVE = "/Volumes/workspace/just_trial/factory_raw_data"

print("1. Scansione globale del Volume alla ricerca di file JSON con Unity Catalog...")

try:
    tabella_controllo = f"{CATALOGO_TARGET}.{SCHEMA_TARGET}.{PREFISSO_TABELLE}_root"
    
    file_gia_caricati = set()
    if spark.catalog.tableExists(tabella_controllo):
        print(f"Estrazione file già elaborati dalla tabella {tabella_controllo}...")
        df_log = spark.table(tabella_controllo).select("nome_file_origine").distinct()
        file_gia_caricati = set([row.nome_file_origine for row in df_log.collect()])
    
    all_files_df = spark.read.format("json").load(PATH_VOLUME_ROOT_RECURSIVE) \
        .select(
            element_at(split(col("_metadata.file_path"), "/"), -1).alias("nome_file_completo"), 
            col("_metadata.file_path").alias("percorso_completo")
        ) \
        .distinct()
    
    lista_totale_files = all_files_df.collect()
    
    percorsi_file_nuovi = [row.percorso_completo for row in lista_totale_files if row.nome_file_completo not in file_gia_caricati]
    
    if not percorsi_file_nuovi:
        print("✓ Nessun file nuovo trovato nell'intero storico del Volume. Il database è già aggiornato.")
    else:
        print(f"Trovati {len(percorsi_file_nuovi)} file nuovi/arretrati da elaborare!")
        
        df_nuovi = spark.read \
            .option("multiline", "true") \
            .json(percorsi_file_nuovi) \
            .withColumn("nome_file_origine", element_at(split(col("_metadata.file_path"), "/"), -1)) \
            .withColumn("file_loading_dtm", current_timestamp())
            
        print("2. Esplosione ricorsiva dei nuovi file...")
        
        dizionario_tabelle = appiattisci_json_relazionale(
            df_nuovi,
            chiavi_padre=["nome_file_origine", "file_loading_dtm", "id_azienda"],
            nome_livello_precedente="root_summaries"
        )
        
        print("3. Scrittura incrementale nel catalogo Delta...")
        print("-" * 60)
        
        for nome_struttura, df_tabella in dizionario_tabelle.items():
            nome_tabella_finale = f"{CATALOGO_TARGET}.{SCHEMA_TARGET}.{PREFISSO_TABELLE}_{nome_struttura}"
            
            print(f"Scrittura in corso su: {nome_tabella_finale}...")
            
            df_tabella.write \
                .format("delta") \
                .mode("append") \
                .option("mergeSchema", "true") \
                .saveAsTable(nome_tabella_finale)
                
            print(f"✓ Righe aggiunte a {nome_tabella_finale}: {df_tabella.count()}")

except Exception as e:
    print(f"✗ Errore critico durante l'elaborazione globale: {str(e)}")

print("-" * 60)
print("Processo di allineamento storico completato!")

In [0]:
%sql
select * from workspace.just_trial.fact_stats_dipartimenti

In [0]:
%sql
select * from workspace.staging.sportradar_root

In [0]:
%sql
select table_name
from information_schema.columns
where column_name='id_azienda'

In [0]:
catalogo = "workspace"
schema = "just_trial"

# Ottiene la lista di tutti gli oggetti nello schema
oggetti = spark.catalog.listTables(f"{catalogo}.{schema}")

for o in oggetti:
    nome_completo = f"{catalogo}.{schema}.{o.name}"
    
    # Controlliamo il tipo: cancelliamo solo Tabelle (MANAGED/EXTERNAL) e Viste
    if o.tableType in ["MANAGED", "EXTERNAL", "VIEW"]:
        spark.sql(f"DROP TABLE IF EXISTS {nome_completo}")
        spark.sql(f"DROP VIEW IF EXISTS {nome_completo}")
        print(f"✗ Eliminata: {nome_completo} ({o.tableType})")

print("--- Pulizia completata! I Volumi non sono stati toccati. ---")

In [0]:
%sql
with step1 as (
    select distinct lists_id
    from workspace.staging.sportradar_lists
),

step2 as (
    select distinct lists_id
    from workspace.staging.sportradar_datapoints
)

select *
from step1 as a 
full outer join step2 as b
    on a.lists_id=b.lists_id;